In [1]:
import pandas as pd
import os
import platform
import json
# from docx import Document
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import matplotlib.pyplot as plt
import seaborn as sns


In [15]:
def preprocess_segments(Dir_github,story,shuffle):
    # Dir_segments = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github\data\segmentation'
    # file = f"segment_{story}_formatted.csv"
    # filepath = os.path.join(Dir_segments,file)
    Dir_segments = os.path.join(Dir_github,'derivatives','behavioral','segmentation')

    file = f"segment_{story}_formatted.csv"
    filepath = os.path.join(Dir_segments,file)
    df = pd.read_csv(filepath)
    df['segment'] = df['segment'].str.strip().replace('\n','')
    df['idx_raw'] = df['idx']
    df['segment_raw'] = df['segment']
    # remove special symbols
    for i,row in df.iterrows():
        line = row['segment']
        while (line[-1] in '",.…!?”“') | (line.endswith(' ')) :
           line = line[0:-1]
        df.loc[i,'segment'] = line

    # shuffle if needed
    if shuffle == 1:
        df.sample(frac=1).reset_index(drop=True)
        df = df.sample(frac=1).reset_index(drop=True)
        df['idx'] = df.index

    # add index
    df['text'] = df['idx']+1
    df['text'] = df['text'].astype(str)
    df['text'] = '#####'+df['text'] + ' ' + df['segment']+'#####\n'

    N_segments = df.shape[0]
    lines_story = ' '.join(df['text'].values)
    return(df,lines_story,N_segments)

def check_alignment(df_input,df):
    flag = 1

    if df_input.shape[0]!=df.shape[0]:
        flag=-1

    df_input = pd.merge(left=df_input,right=df,left_on='idx',right_on='idx',how='left')
    
    for i,row in df_input.iterrows():
        flag1 = '0'
        flag2 = '0'
        input = row['segment']
        word1 = str(row['firstword'])
        word2 = str(row['lastword'])
        if input.startswith(word1):
            flag1 = '1'
        if input.endswith(word2):
            flag2 = '1'
        df_input.loc[i,'flag'] = flag1+flag2
    return(df_input,flag)

def format_ratings_gpt(story,file):
    f = open(file)
    data = json.load(f)
    f.close()
    List_lines = data['choices'][0]['message']['content'].replace('"""','').split('\n')
    List_lines = [x for x in List_lines if x!='']
    df = pd.DataFrame(List_lines,columns=['raw'])
    df[['V','A']] = df['raw'].str.rsplit(' ',n=2,expand=True)[[1,2]]
    df['idx'] = df['raw'].str.split(' ',n=1,expand=True)[0]
    df['firstlast'] = df['raw'].str.rsplit(' ',n=2,expand=True)[0]
    df['firstlast'] = df['firstlast'].str.split(' ',n=1,expand=True)[1]
    if df['firstlast'].str.split(' ',n=1,expand=True).shape[1] == 1:
        df['firstword'] = df['firstlast']
        df['lastword'] = 'ERROR:NOT EXPORT LAST WORD'
    else:
        df[['firstword','lastword']] = df['firstlast'].str.split(' ',n=1,expand=True)
    df['idx'] = df['idx'].astype(int)
    return(df)
    
def get_rating_gpt(Dir_github,model,List_story):
    Dir_results = os.path.join(Dir_github,'derivatives','behavioral','ratings','gpt',model)
    List_files_all = os.listdir(Dir_results)

    dfs_score = []
    for story in List_story:
        if 'gpt-4o' in model:
            List_files = [x for x in List_files_all if '.csv' in x and '_query' not in x and story in x]
        else:
            List_files = [x for x in List_files_all if '.json' in x and '_query' not in x and story in x]
        if len(List_files)==0:
            print(f"{story}: no results")
    
        # import raw input to check alignment
        df_input,lines_story,N_segs = preprocess_segments(Dir_github,story,0)
        df_input['idx'] = df_input['idx']+1
    
        print(story,N_segs)
        for iter,file in enumerate(List_files):
            
            # import one iteration
            if 'gpt-4o' in model:
                df = pd.read_csv(os.path.join(Dir_results,file))
            else:
                df = format_ratings_gpt(story,os.path.join(Dir_results,file))
            # List_texts = data['choices'][0]['message']['content'].replace('\n','').split('"""')
            
            df_score,flag = check_alignment(df_input,df)
            if flag<0:
                print(f"{story}-iter{iter}:check alignment, incorrect N.")
                continue
            
            df_score.loc[df_score['flag']!='11',['V','A']] = np.nan
            df_score['story'] = story
            df_score['iteration'] = iter
            df_score['idx'] = df_score['idx'].astype(int)
            df_score['V'] = df_score['V'].astype(float)
            df_score['A'] = df_score['A'].astype(float)
            df_score['rater'] = model
            dfs_score.append(df_score)
    
    dfs_score = pd.concat(dfs_score,ignore_index=True)
    dfs_score.to_csv(os.path.join(Dir_results,f"rating_gpt_{model}.csv"),index=False)
    return(dfs_score)

In [3]:
if platform.system() == 'Linux':
    Dir_github = r'/work/xy6/Project/ABC/github/' # HPC
else:
    Dir_github = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github' # Local computer
repo = 'ABC_valence'
pipeline = 'getRatings_GPT'
Dir_working = os.path.join(Dir_github,repo,'scripts',pipeline)

Dir_segments = os.path.join(Dir_github,'derivatives','behavioral','segmentation')
df_story = pd.DataFrame({'story':[x for x in os.listdir(Dir_segments) if '_formatted.csv' in x ]})
df_story['story'] = df_story['story'].str.split('_',expand=True)[1]
for i,row in df_story.iterrows():
    df_tmp = pd.read_csv(os.path.join(Dir_segments,f"segment_{row['story']}_formatted.csv"))
    df_story.loc[i,'N_segments'] = df_tmp.shape[0]
df_story.sort_values(['story'],inplace=True,ignore_index=True)
List_story = df_story['story'].values

In [16]:
for model in ['gpt-4-1106-preview','gpt-4o-2024-08-06','gpt-4o-2024-11-20']:
    print(f'========> {model}')
    dfs_score = get_rating_gpt(Dir_github,model,List_story)

========> gpt-4-1106-preview
AgedMother 128
AgedMother-iter8:check alignment, incorrect N.
Aqua 147
Aqua-iter1:check alignment, incorrect N.
Aqua-iter2:check alignment, incorrect N.
Aqua-iter5:check alignment, incorrect N.
Aqua-iter6:check alignment, incorrect N.
Aqua-iter7:check alignment, incorrect N.
Aqua-iter8:check alignment, incorrect N.
Aqua-iter9:check alignment, incorrect N.
BiggestWastes 152
BiggestWastes-iter7:check alignment, incorrect N.
BradfordBoar 157
BradfordBoar-iter0:check alignment, incorrect N.
BradfordBoar-iter1:check alignment, incorrect N.
BradfordBoar-iter2:check alignment, incorrect N.
BradfordBoar-iter3:check alignment, incorrect N.
BradfordBoar-iter5:check alignment, incorrect N.
BradfordBoar-iter6:check alignment, incorrect N.
BradfordBoar-iter7:check alignment, incorrect N.
BradfordBoar-iter8:check alignment, incorrect N.
BradfordBoar-iter9:check alignment, incorrect N.
BulliesPronoun 158
BulliesPronoun-iter2:check alignment, incorrect N.
BulliesPronoun-it